In [4]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [14]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000] , y_train_full[5000:]

In [15]:
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=[28 , 28]))
model.add(keras.layers.Dense(300, activation="relu"))
model.add(keras.layers.Dense(100, activation="relu"))
model.add(keras.layers.Dense(10, activation="softmax"))

model.summary()

c:\Users\gunng\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 266,610 (1.02 MB)

 Trainable params: 266,610 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])

In [17]:
history = model.fit(X_train, y_train, epochs=30, validation_data=(X_valid, y_valid))

Epoch 1/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7682 - loss: 0.7089 - val_accuracy: 0.8060 - val_loss: 0.5470
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8300 - loss: 0.4896 - val_accuracy: 0.8422 - val_loss: 0.4512
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8441 - loss: 0.4466 - val_accuracy: 0.8504 - val_loss: 0.4202
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8522 - loss: 0.4179 - val_accuracy: 0.8594 - val_loss: 0.4129
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8596 - loss: 0.3982 - val_accuracy: 0.8620 - val_loss: 0.3965
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8657 - loss: 0.3826 - val_accuracy: 0.8524 - val_loss: 0.4071
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8681 - loss: 0.3695 - val_accuracy: 0.8730 - val_loss: 0.3645
Epoch 8/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8736 - loss: 0.3561 - 

In [18]:
model.save("my_model.keras")

# Load it back
reconstructed_model = keras.models.load_model("my_model.keras")

## Transfer Learning with keras

In [19]:
model_A = keras.models.load_model("my_model.keras")
model_B_on_A = keras.models.Sequential(model_A.layers[:-1])
model_B_on_A.add(keras.layers.Dense(1, activation="sigmoid"))

we also can use model_clone to get the model A's archtecture but you have to add the weights after that

In [20]:
model_A_clone = keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

In [22]:
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = False

model_B_on_A.compile(loss="binary_crossentropy", optimizer="sgd", metrics=["accuracy"])

In [23]:
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=4, validation_data=(X_valid_B, y_train_B))

for layer in model_B_on_A.layers[:-1]:
    layer.trainable = True

optimizer = keras.optimizers.SGD(lr=1e-4)
model_B_on_A.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=16, validation_data=(X_valid_B, y_valid_B))

NameError: name 'X_train_B' is not defined

Transfer Learning while it can yielded a better result, most of the time, the improvement will generally drops or even reverse.

## Unsupervised Pretraining

TLDR: labeled data are expensive and rare to find, you can use lower layers of unsupervised models such as GANs or autoencoder and add the output layer for the task on the top and fine tune the final network using supervised learning

## Faster Optimizers

1. Momentum Optimization -> use concept of momentum to make it faster than normal gradient descent. NEW PARAMETER FOR THIS IS momentum (this is for adding some fiction to the momentum optimizer, so that it will not overshoot back and forth)

In [ ]:
optimizer = keras.optimizers.SGD(lr=0.001, momentum = 0.9)

2. Nesterov Accelerated Gradient -> better than vanilla momentum optimizer because it measures the gradient of the cost funciton not at the local position. Make it converages faster.

In [1]:
optimizer = keras.optimizers.SGD(lr=0.001, momentum = 0.9, nesterov=True)

NameError: name 'keras' is not defined

3. AdaGrad -> it can change the learning rate make the updates more directly toward the global optimum. BUT!! it generally stop too early and does not reach the global optimum. So, it is not recommended to use it for training neural network

4. RMSProp -> fix the AdaGrad problem (stop too fast, never reach global optimum) by accumulating only the gradients from the most recent iterations. NEW PARAMETER, rho -> for the decay rate

In [ ]:
optimizer = keras.optimizers.SGD(lr=0.001, momentum = 0.9, rho=0.9)

5. Adam (Adaptive moment estimation) -> combine ideas from both momentum optimization and RMSProps to both track the exponentially decaying avg of past gradients while keeping track of exponentially decaying avg of past squared gradients

In [ ]:
optimizer = keras.optimizers.Adam(lr=0.001, beta_1=0.9, beta_2=0.999)

PS!! most of the times the adaptive optimization are great, but in some cases, using plain Nesterov Accelerated Gradient but perform better when the dataset is allergic to adaptive gradients.